# Notebook 03 — Teach SmolLM2 maths with LoRA

Train a small adapter on top of the original `HuggingFaceTB/SmolLM2-360M`. The original model weights stay frozen; the adapter learns from the worked GSM8K solutions and their labelled final answers.

**Kaggle setup:** enable Internet, select a T4 GPU, and attach the **original saved Notebook 01 output** using Add Input. One GPU is used even if Kaggle supplies two T4s. You do not need to attach the pilot output for training; retain it for the later comparison.

This notebook audits tokenization, runs CPU contract tests, runs a two-update GPU smoke check, and then starts a fresh three-epoch training run. Checkpoints are selected using **validation response-token loss**, with no test-score selection. The saved adapter is not a standalone full model. Lower validation loss does not by itself prove higher generated maths accuracy; paired evaluation comes next.


## 1. Fetch the shared code

The launcher records the exact GitHub commit. Training logic and settings live in the repository. Keep the original data bundle fixed throughout the experiment.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

## 2. Verify the original bundle and inspect the settings

The first run uses rank 16 LoRA on attention `q_proj`, `k_proj`, `v_proj`, and `o_proj`, learning rate `2e-4`, and three epochs. A micro-batch of four examples accumulated over eight micro-batches gives an effective batch of 32, except the last partial update of an epoch.

Training is ordinary LoRA with an FP16 frozen base and FP32 trainable adapters. No quantization or chat template is introduced. The supplied bundle records the original model revision and train/validation split.


In [ ]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before continuing.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
CONFIG_PATH = PROJECT / "configs/lora_training.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
SOURCE_MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
CONFIG_HASH = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
RUN_ROOT = Path("/kaggle/working") / f"lora-v1-{BUNDLE_HASH[:12]}-{CONFIG_HASH[:8]}-{CODE_COMMIT[:8]}"
SMOKE_OUTPUT, TRAIN_OUTPUT = RUN_ROOT / "smoke", RUN_ROOT / "full"
print("Bundle:", BUNDLE)
print("Pinned model:", SOURCE_MANIFEST["model"])
print("Original data counts:", SOURCE_MANIFEST["counts"])
print("Training settings:", json.dumps(CONFIG, indent=2))
print("Output:", RUN_ROOT)


## 3. Prepare the training environment

This adds pinned PEFT and Accelerate packages to the known isolated PyTorch setup. The Kaggle kernel packages are not replaced. The shared setup avoids Kaggle's missing `ensurepip` problem.

The existing `wrapt` startup warning and `torch_dtype` deprecation warning may appear. CUDA execution and the following contract tests must pass before training.


In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-training-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
TRAIN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
probe = """
import torch, peft, accelerate
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: check Kaggle GPU settings.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
print('PEFT:', peft.__version__, 'Accelerate:', accelerate.__version__)
print('Visible GPUs for training:', torch.cuda.device_count())
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True, env=TRAIN_ENV)


## 4. Verify the training contracts on CPU

Tiny local models check prompt/padding masking, EOS supervision, exact response-token loss, frozen base weights, validation-based selection, adapter reload, checkpoint recovery, and refusal to mix changed experiments. These tests do not download SmolLM2 or claim a maths improvement.


In [ ]:
_ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_training.py", "-b", "-v"],
                    cwd=PROJECT, check=True, env=TRAIN_ENV)


## 5. Audit training and validation tokenization before loading weights

Each example is encoded as separately tokenized prompt + response + EOS. Prompt labels and padding are `-100`, so the loss trains the solution and final answer. Prompt text remains visible as context. There is no packing or truncation.

Every training token count must match Notebook 01's saved audit. Validation solutions are teacher-forced in the same zero-shot text format used for training. This is different from generating answers: it measures how well the model predicts the supplied correct solution tokens.

All original training/validation rows are used in the full run. If a sequence exceeds 1024 tokens, stop and inspect it rather than silently dropping or shortening it.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "full",
                    "--output", str(RUN_ROOT / "audit"), "--audit-only"],
                    check=True, cwd=PROJECT, env=TRAIN_ENV)
AUDIT = json.loads((RUN_ROOT / "audit/data_audit.json").read_text())
print({key: AUDIT[key] for key in ("train_rows", "validation_rows", "max_train_tokens", "max_validation_tokens", "effective_batch_size", "planned_optimizer_updates")})


## 6. Run two GPU optimizer updates as an engineering check

Use the first 64 training rows and 16 validation rows. This checks real model loading, LoRA gradients, finite losses/parameters, checkpoint saving, and an actual adapter change. It is not a useful trained model or an accuracy benchmark.

The full run starts again from the original model with a fresh adapter. It does not continue from these smoke weights. If this cell fails, inspect its error before running the full-training cell.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "smoke",
                    "--output", str(SMOKE_OUTPUT)], check=True, cwd=PROJECT, env=TRAIN_ENV)
SMOKE = json.loads((SMOKE_OUTPUT / "summary.json").read_text())
if not SMOKE["health_passed"]:
    raise RuntimeError("Training smoke check failed; inspect the saved outputs.")
print("Smoke check:", json.dumps(SMOKE, indent=2))


## 7. Train the full adapter and select by validation loss

Train for three epochs. Evaluate and save every 100 optimizer updates and at epoch ends, keeping the best checkpoint plus recent recovery state. The selected adapter has the lowest mean loss over the supervised validation response tokens, including EOS. Short solutions are not given the same token weight as much longer ones.

The trainer records the initial base-model validation loss and all checkpoint losses. A decreasing training loss can accompany overfitting, so validation loss chooses the saved adapter. Neither loss establishes test accuracy or separates formatting gains from mathematical gains.

If interrupted, rerun this cell in the same session. It resumes the latest completed checkpoint, including optimizer/scheduler/random state. Changes to code, packages, configuration, bundle, or runtime are refused. Uncheckpointed updates must be repeated. Save the Kaggle outputs before the session ends; runtime identity checks may prevent continuing in a different environment.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "full",
                    "--smoke-output", str(SMOKE_OUTPUT), "--output", str(TRAIN_OUTPUT)],
                    check=True, cwd=PROJECT, env=TRAIN_ENV)
SUMMARY = json.loads((TRAIN_OUTPUT / "summary.json").read_text())
print(json.dumps(SUMMARY, indent=2))


## 8. Review and save the adapter

`full/adapter` contains the selected adapter, not a merged copy of the 360M base model. Keep the complete ZIP, tokenizer, identity, base revision, config, validation history, and manifests. The archive also retains smoke results and recovery checkpoints.

For the next evaluation notebook, attach this saved output, the original Notebook 01 bundle, and the saved Notebook 02c pilot output. Both models must use the same 300 questions, prompts, stopping, and official scoring. Also inspect answer-format compliance and the documented arithmetic audit: printing `Final answer:` more reliably can improve the official score without an equivalent gain in mathematical ability.


In [ ]:
from IPython.display import HTML, display
for output in (SMOKE_OUTPUT, TRAIN_OUTPUT):
    result = json.loads((output / "result_manifest.json").read_text())
    for name, expected in result["files_sha256"].items():
        if hashlib.sha256((output / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Training result checksum mismatch: {name}")
display(HTML((TRAIN_OUTPUT / "report.html").read_text()))
ARCHIVE = Path(shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT))
print("Selected adapter:", TRAIN_OUTPUT / "adapter")
print("Training ZIP:", ARCHIVE)
print("Save Version with outputs. Next: paired evaluation against the saved baseline pilot.")
